In [1]:
import pandas as pd
import numpy as np
import pyarrow
from src_RN import *
import itertools, gc

from sklearn.model_selection import  train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, MinMaxScaler
from sklearn.model_selection import StratifiedKFold

import keras
from keras.models import Sequential
from keras.layers import Dense, Input, Dropout
from keras import optimizers

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adam

# Configurações 
print(tf.config.list_physical_devices())
BATCH_SIZE = 64

KeyboardInterrupt: 

# 1.0 - Classificação de Presença Baseada em Fatores Socioeconômicos Usando Redes Neurais

In [ ]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_SEXO','TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df = pd.read_parquet("enem_parquet", columns = colunas)

## 1.2- Pré-Processando os Dados

In [ ]:
df = pre_processor_rn(df, objetivo = 'Desempenho', n_samples = 50000)

## 1.3- Construção da Matriz X e Vetor y

In [ ]:
X = df.drop(columns=['MEDIA', 'CLASSE', 'FALTOU'])
y = df['CLASSE']

## 1.4 - Separação em Dados de Treino, Validação e Teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

## 1.5 - Tratando os Dados

In [ ]:
preprocessador = transformar(X_train)

X_train = preprocessador.transform(X_train).astype(np.float32)
X_val   = preprocessador.transform(X_val).astype(np.float32)
X_test  = preprocessador.transform(X_test).astype(np.float32)

## 1.6 - Treinando a Rede Neural

In [ ]:
max_neurons = num_max_neuronio(X_train, d = X_train.shape[1])
print("Número máximo de neurônios:", max_neurons)

Número máximo de neurônios: 14


In [ ]:
param_grid = {
    'neurons':       [13],
    'learning_rate': [0.1, 0.01, 0.001, 0.0001],   
    'batch_size':    [32,64],             
    'epochs':        [100],
    'l2_reg':        [0.001, 0.01, 0.1],          
    'dropout':       [0.0, 0.2],
}

# Gera todas as combinações
keys, values = zip(*param_grid.items())
combinacoes = [dict(zip(keys, v)) for v in itertools.product(*values)]
print(f"Total de combinações: {len(combinacoes)}")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

resultados = []

for params in combinacoes:
    print(f"Testando: {params}")
    
    accs = []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val_fold = X_train[train_idx], X_train[val_idx]
        y_tr, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = create_model(
            input_dim=X_train.shape[1],
            neurons=params['neurons'],
            learning_rate=params['learning_rate'],
            l2_reg=params['l2_reg'],
            dropout=params['dropout']
        )

        early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_val_fold, y_val_fold),
            epochs=params['epochs'],  
            batch_size=params['batch_size'],
            callbacks=[early_stop],
            verbose=1
        )

        # Avaliação no fold
        loss, acc = model.evaluate(X_val_fold, y_val_fold, verbose=0)
        accs.append(acc)

        # Libera memória (importante no Keras)
        del model
        gc.collect()

    mean_acc = np.mean(accs)

    resultados.append({
        'params': params,
        'mean_accuracy': mean_acc
    })

    print(f"Accuracy média: {mean_acc:.4f}")

Total de combinações: 48
Testando: {'neurons': 13, 'learning_rate': 0.1, 'batch_size': 32, 'epochs': 100, 'l2_reg': 0.001, 'dropout': 0.0}


C:\Users\Micael\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6743 - loss: 0.7329 - val_accuracy: 0.6849 - val_loss: 0.6453
Epoch 2/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6916 - loss: 0.6150 - val_accuracy: 0.6978 - val_loss: 0.5977
Epoch 3/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6919 - loss: 0.6055 - val_accuracy: 0.7012 - val_loss: 0.5929
Epoch 4/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6824 - loss: 0.6086 - val_accuracy: 0.6926 - val_loss: 0.6111
Epoch 5/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6822 - loss: 0.6088 - val_accuracy: 0.6691 - val_loss: 0.6261
Epoch 6/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6745 - loss: 0.6160 - val_accuracy: 0.6875 - val_loss: 0.6175
Epoch 7/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6747 - loss: 0.6221 - val_accuracy: 0.6875 - val_loss: 0.6046
Epoch 8/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6887 - loss: 0.6132 - val_accu